In [ ]:

# Statistical Analysis: Customer Transactions
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('transactions.csv')
print(df.head())

In [ ]:

# Step 1: Descriptive Statistics
print("Basic Statistics:")
print(f"Count: {df['amount'].count()}")
print(f"Mean: ${df['amount'].mean():.2f}")
print(f"Median: ${df['amount'].median():.2f}")
print(f"Mode: ${df['amount'].mode()[0]:.2f}")
print(f"Std Dev: ${df['amount'].std():.2f}")
print(f"Variance: ${df['amount'].var():.2f}")
print(f"Min: ${df['amount'].min():.2f}")
print(f"Max: ${df['amount'].max():.2f}")
print(f"Range: ${df['amount'].max() - df['amount'].min():.2f}")

In [ ]:

# Step 2: Percentiles and Quartiles
percentiles = df['amount'].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
print("\nPercentiles:")
print(f"Q1 (25th): ${percentiles[0.25]:.2f}")
print(f"Q2 (50th/Median): ${percentiles[0.50]:.2f}")
print(f"Q3 (75th): ${percentiles[0.75]:.2f}")
print(f"P90: ${percentiles[0.90]:.2f}")
print(f"P95: ${percentiles[0.95]:.2f}")
print(f"P99: ${percentiles[0.99]:.2f}")

df['amount'].plot(kind='box', figsize=(8, 6))
plt.title('Transaction Amount Distribution')
plt.ylabel('Amount ($)')
plt.show()

In [ ]:

# Step 3: Outlier Detection (IQR Method)
q1 = df['amount'].quantile(0.25)
q3 = df['amount'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df[(df['amount'] < lower_bound) | (df['amount'] > upper_bound)]
print(f"\nOutlier Count: {len(outliers)}")
print(f"Outlier Percentage: {len(outliers) / len(df) * 100:.2f}%")

df_clean = df[(df['amount'] >= lower_bound) & (df['amount'] <= upper_bound)]
print(f"Mean without outliers: ${df_clean['amount'].mean():.2f}")

In [ ]:

# Step 4: Correlation Analysis
correlation_matrix = df[['amount', 'customer_age', 'days_since_signup']].corr()
print("\nCorrelation Matrix:")
print(correlation_matrix)

print(f"\nAmount-Age Correlation: {df['amount'].corr(df['customer_age']):.3f}")
print(f"Amount-Tenure Correlation: {df['amount'].corr(df['days_since_signup']):.3f}")

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:

# Step 5: Frequency Distribution (Binning)
bins = [0, 50, 100, 200, 500, float('inf')]
labels = ['0-50', '50-100', '100-200', '200-500', '500+']
df['amount_range'] = pd.cut(df['amount'], bins=bins, labels=labels)

freq_dist = df.groupby('amount_range').agg({
    'amount': ['count', 'mean']
}).round(2)
freq_dist.columns = ['frequency', 'avg_in_range']
freq_dist['percentage'] = (freq_dist['frequency'] / freq_dist['frequency'].sum() * 100).round(2)
print("\nFrequency Distribution:")
print(freq_dist)

df['amount'].hist(bins=20, figsize=(10, 6), color='skyblue', edgecolor='black')
plt.title('Transaction Amount Distribution')
plt.xlabel('Amount ($)')
plt.ylabel('Frequency')
plt.show()

In [ ]:

# Step 6: Z-Score Analysis
df['z_score'] = stats.zscore(df['amount'])
df['anomaly_flag'] = pd.cut(df['z_score'].abs(),
                             bins=[0, 2, 3, float('inf')],
                             labels=['Normal', 'Moderate', 'Extreme'])

anomalies = df[df['z_score'].abs() > 2].sort_values('z_score', ascending=False, key=abs)
print(f"\nAnomalies Detected: {len(anomalies)}")
print(anomalies[['transaction_id', 'amount', 'z_score', 'anomaly_flag']].head(10))

plt.figure(figsize=(12, 6))
plt.scatter(df.index, df['z_score'], c=df['z_score'].abs(), cmap='Reds', alpha=0.6)
plt.axhline(y=2, color='orange', linestyle='--', label='±2 SD')
plt.axhline(y=-2, color='orange', linestyle='--')
plt.axhline(y=3, color='red', linestyle='--', label='±3 SD')
plt.axhline(y=-3, color='red', linestyle='--')
plt.title('Z-Score Distribution - Anomaly Detection')
plt.ylabel('Z-Score')
plt.xlabel('Transaction Index')
plt.legend()
plt.show()